# HR attrition dataset

- EmployeeID (integer, unique)

- Age (integer)

- Department (categorical: Sales, IT, HR, Finance)

- MonthlyIncome (float)

- YearsAtCompany (integer)

 - Attrition (boolean: Yes/No)

- LastPromotionDate (date)

- SatisfactionScore (integer 1–5)

## Generate Clean Data

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n = 500  # number of rows

data = {
    "EmployeeID": range(1001, 1001 + n),
    "Age": np.random.randint(22, 65, n),
    "Department": np.random.choice(["Sales", "IT", "HR", "Finance"], n),
    "MonthlyIncome": np.random.uniform(3000, 12000, n).round(2),
    "YearsAtCompany": np.random.randint(0, 30, n),
    "Attrition": np.random.choice(["Yes", "No"], n, p=[0.2, 0.8]),
    "LastPromotionDate": [datetime(2020,1,1) + timedelta(days=np.random.randint(0, 1500)) for _ in range(n)],
    "SatisfactionScore": np.random.randint(1, 6, n)
}

df_clean = pd.DataFrame(data)

In [14]:
df_clean

,EmployeeID,Age,Department,MonthlyIncome,YearsAtCompany,Attrition,LastPromotionDate,SatisfactionScore
0,1001,60,IT,4903.03,1,No,2022-12-29,5
1,1002,50,Sales,5947.48,7,Yes,2023-05-19,5
2,1003,36,HR,4077.86,26,No,2023-01-29,3
3,1004,64,Finance,11014.75,12,Yes,2021-11-28,4
4,1005,29,Finance,8342.33,12,No,2020-05-11,1
...,...,...,...,...,...,...,...,...
495,1496,30,Finance,7456.32,27,No,2021-07-12,4
496,1497,45,IT,7325.28,14,No,2021-06-10,1
497,1498,56,HR,8331.67,14,No,2021-10-20,3
498,1499,56,Sales,10422.13,5,No,2020-01-26,2


## Inject at least 30% “Dirty” Data

| Error type                  | Example in data                                |
|-----------------------------|------------------------------------------------|
| Missing values              | `None` in Age, SatisfactionScore               |
| Outliers                    | MonthlyIncome = $500,000                       |
| Typos / inconsistent text   | “Saless”, “IT ”, “it”                          |
| Wrong date format           | “2025-13-01” or “Jan 1, 2025”                  |
| Negative values             | Age = -5, YearsAtCompany = -2                  |
| Duplicate rows              | Two rows with same EmployeeID                  |
| Numeric as text             | “three thousand” instead of 3000               |
| Boolean as “Y/N” / “T/F”    | Attrition = “Y”/“N” instead of Yes/No          |

In [15]:
df_dirty = df_clean.copy()

In [47]:
# 1. Missing values (25% of rows in selected columns)
import numpy as np  
for col in ["Age", "SatisfactionScore"]:
    mask = np.random.random(len(df_dirty)) < 0.25
    df_dirty.loc[mask, col] = None

In [48]:
# 2. Typos in Department (15% of entries)
typo_map = {"Sales": "Saless", "IT": "It ", "HR": "H R", "Finance": "Finence"}
for correct, wrong in typo_map.items():
    mask = (df_dirty["Department"] == correct) & (np.random.random(len(df_dirty)) < 0.15)
    df_dirty.loc[mask, "Department"] = wrong

In [49]:
# 3. Outliers in MonthlyIncome (5% > 200,000)
mask = np.random.random(len(df_dirty)) < 0.05
df_dirty.loc[mask, "MonthlyIncome"] = np.random.uniform(200000, 500000, mask.sum())

In [50]:
# 4. Negative YearsAtCompany (13%)
mask = np.random.random(len(df_dirty)) < 0.23
df_dirty.loc[mask, "YearsAtCompany"] = -np.random.randint(1, 10, mask.sum())

In [51]:
# 5. Wrong date format (25%)
# First convert to object to allow string + datetime mixed
df_dirty["LastPromotionDate"] = df_dirty["LastPromotionDate"].astype(object)

mask = np.random.random(len(df_dirty)) < 0.25
df_dirty.loc[mask, "LastPromotionDate"] = "2025-13-45"   #

In [52]:
# 6. Duplicates – append 20 duplicate rows from random indices
duplicates = df_dirty.sample(n=30, replace=True)
df_dirty = pd.concat([df_dirty, duplicates], ignore_index=True)

In [53]:
# 7. Boolean as "Y"/"N" (instead of Yes/No)
mask = np.random.random(len(df_dirty)) < 0.1
df_dirty.loc[mask, "Attrition"] = df_dirty.loc[mask, "Attrition"].map({"Yes":"Y", "No":"N"})


In [54]:
# 8. Numeric stored as text (Age column, 5%)
df_dirty["Age"] = df_dirty["Age"].astype(object)
mask = np.random.random(len(df_dirty)) < 0.05
df_dirty.loc[mask, "Age"] = df_dirty.loc[mask, "Age"].apply(lambda x: str(x) + " years")

In [55]:
def is_dirty_cell(col, val):
    if pd.isna(val):
        return True
    if col == "YearsAtCompany" and isinstance(val, (int, float)) and val < 0:
        return True
    if col == "MonthlyIncome" and isinstance(val, (int, float)) and val > 200000:
        return True
    if col == "Department" and isinstance(val, str) and val in ["Saless", "It ", "H R", "Finence"]:
        return True
    if col == "Age" and isinstance(val, str) and "years" in val:
        return True
    if col == "LastPromotionDate" and val == "2025-13-45":
        return True
    if col == "Attrition" and val in ["Y", "N"]:
        return True
    return False

total_cells = 0
dirty_cells = 0
for col in df_dirty.columns:
    for val in df_dirty[col]:
        total_cells += 1
        if is_dirty_cell(col, val):
            dirty_cells += 1

print(f"Total cells: {total_cells}")
print(f"Dirty cells: {dirty_cells}")
print(f"Dirty ratio: {dirty_cells/total_cells:.1%}")

Total cells: 5200
Dirty cells: 2116
Dirty ratio: 40.7%


In [59]:
df_dirty.to_csv(r'C:\Users\iions\Documents\New folder\dylitica _assignment\hr_attrition_dirty_20260527_203832.csv', index=False)
